In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [71]:
df = pd.read_csv('analysed_data.csv')
df.head()

,area,room,living_room,age,floor,city,district,neighbourhood,price
0,90,2,1,6,6,istanbul,kagithane,caglayan,39500
1,75,2,1,29,2,istanbul,kagithane,ortabayir,28000
2,95,2,1,18,2,istanbul,maltepe,aydinevler,46000
3,65,2,1,9,0,istanbul,kucukcekmece,kemalpasa,22000
4,145,3,1,40,2,istanbul,sisli,mesrutiyet,95000


In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6059 entries, 0 to 6058
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   area           6059 non-null   int64 
 1   room           6059 non-null   int64 
 2   living_room    6059 non-null   int64 
 3   age            6059 non-null   int64 
 4   floor          6059 non-null   int64 
 5   city           6059 non-null   object
 6   district       6059 non-null   object
 7   neighbourhood  6059 non-null   object
 8   price          6059 non-null   int64 
dtypes: int64(6), object(3)
memory usage: 426.2+ KB


In [73]:
print(df.columns.tolist())

['area', 'room', 'living_room', 'age', 'floor', 'city', 'district', 'neighbourhood', 'price']


In [74]:
categorical_features = ['district', 'neighbourhood', 'city']
numerical_features = ['area', 'room', 'living_room', 'age', 'floor']


In [75]:
full_pipeline = ColumnTransformer([
    ('num', StandardScaler(), numerical_features), # Standart Scaler used to scale numerical features
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) # One Hot Encoder used to encode categorical features
])

In [76]:
X = df.drop(columns=['price'], axis=1)
y = df['price']

In [77]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=31)


In [78]:
model = Pipeline([
    ('preparation', full_pipeline),
    ('model', LinearRegression())])

In [79]:
model.fit(X_train,y_train)

Pipeline(steps=[('preparation',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['area', 'room',
                                                   'living_room', 'age',
                                                   'floor']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['district', 'neighbourhood',
                                                   'city'])])),
                ('model', LinearRegression())])

In [80]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("R^2 Score:", r2)


Mean Squared Error: 125827228.7790465
Root Mean Squared Error: 11217.273678530202
R^2 Score: 0.7003486585949588


In [81]:
feature_importances = model.named_steps['model'].coef_
print("Feature Importances:", feature_importances)

Feature Importances: [ 8.44320137e+03  7.32108357e+02  0.00000000e+00 -6.51115251e+03
  1.86311786e+03  2.03642592e+03 -2.06365556e+04  1.06072393e+04
 -2.19821071e+04 -5.85673007e+03  1.31376539e+04  2.58209485e+04
 -1.85236767e+04 -1.46959681e+02  1.91947068e+04 -1.00657631e+04
 -1.77206507e+03  1.37417811e+04 -7.07419552e+03  2.00274850e+04
 -9.97949025e+03 -6.95429574e+03 -1.80411796e+04 -5.94614078e+03
  8.73440634e+03 -3.02757531e+03  2.27292498e+03  2.29723650e+04
 -3.08654601e+02  9.76709836e+02 -2.59594719e+02  1.25110309e+04
 -4.03578628e+03 -1.34701263e+04  7.76620755e+03  7.29360793e+03
 -1.66900019e+04  2.02787619e+04 -1.62899146e+04 -1.11304270e+04
 -3.63200732e+03 -7.08823285e+03  6.48500194e+03  9.05422329e+03
 -1.08444076e+04 -2.05101108e+03 -7.38847214e+03  3.16961557e+04
  7.71773237e+03 -1.05096935e+04 -5.65691988e+03  6.53917184e+03
  3.16234367e+03  4.00914128e+04  3.63617986e+03 -1.40264130e+04
  1.09940747e+04 -1.28141786e+04  3.52834007e+02 -7.49657081e+03
 -1.

In [82]:
print("Numerical Features")
for i in range(len(numerical_features)):
    print(f"{numerical_features[i]}: {feature_importances[i]}")

Numerical Features
area: 8443.20137477857
room: 732.1083572410548
living_room: 0.0
age: -6511.152507179897
floor: 1863.1178592298284


In [83]:
print("Categorical Features")
for i in range(len(categorical_features)):
    print(f"{categorical_features[i]}: {feature_importances[len(numerical_features) + i]}")

Categorical Features
district: 2036.4259241536076
neighbourhood: -20636.55557365853
city: 10607.239253393038


In [86]:
example_data = {
    'area': [55],
    'room': [1],
    'living_room': [1],
    'age': [10],
    'floor': [2],
    'district': ['kadikoy'],
    'neighbourhood': ['moda'],
    'city': ['istanbul']
}
print(model.predict(pd.DataFrame(example_data)))

[51464.68400072]
